In [32]:
import asyncio
import sys
import time
import re

In [3]:
class Logable:
    """
    Classe de base pour disposer d'une méthode log
    """

    def __init__(self, name, verbose):
        self.name = name
        self.verbose = verbose

    def log(self, msg):
        if self.verbose:
            print(f"[{self.name}] {msg}", file=logf, flush=True)

In [10]:
class Accessoire(list, Logable):

    def __init__(self, name, verbose):
        list.__init__(self)
        Logable.__init__(self, name, verbose)

In [11]:
class Pic(Accessoire):

    def embrocher(self, postit):
        self.log(
            f"post-it '{postit}' embrochée, "
            f"{len(self)} post-it(s) à traiter"
        )
        self.append(postit)

    def liberer(self):
        if len(self) > 0:
            postit = self.pop()

            self.log(
                f"post-it '{postit}' libéré, "
                f"{len(self)} post-it(s) à traiter"
            )

            return postit

        return None

In [18]:
class Bar(Accessoire):

    def recevoir(self, commande):
        self.log(
            f"'{commande}' posée, "
            f"{len(self)} commande(s) à servir"
        )
        self.append(commande)

    def evacuer(self):
        if len(self) > 0:
            commande = self.pop()

            self.log(
                f"'{commande}' évacuée, "
                f"{len(self)} commande(s) à servir"
            )

            return commande

        return None

In [19]:
class Employe(Logable):

    def __init__(self, pic, bar, clients, name, verbose):
        Logable.__init__(self, name, verbose)

        self.pic = pic
        self.bar = bar
        self.clients = clients
        self.step = 0

        self.log("prêt pour le service")

In [27]:
class Clients:

    def __init__(self, fname):
        commandes = []
        start = time.time()
        fmt = re.compile(r"(\d+)\s+(.*)")

        with open(fname, "r", encoding="utf-8") as f:
            for line in f:
                found = fmt.search(line)

                if found:
                    when = int(found.group(1))
                    what = found.group(2)
                    commandes.append((start + when, what.split(",")))

        self.commandes = commandes[::-1]

    async def run(self, serveur):
        while self.commandes:
            when, commande = self.commandes[-1]

            attente = when - time.time()

            if attente > 0:
                await asyncio.sleep(attente)

            self.commandes.pop()
            await serveur.prendre_commande(commande)

In [33]:
class Serveur(Employe):

    async def prendre_commande(self, commande):
        self.log("prêt pour prendre une nouvelle commande...")

        self.log(f"j'ai la commande '{commande}'")
        self.log(f"j'écris sur le post-it '{commande}'")

        self.pic.embrocher(commande)

    async def servir(self):
        commande = self.bar.evacuer()

        if commande is not None:
            self.log(f"j'apporte la commande '{commande}'")

            for cocktail in commande:
                self.log(f"je sers '{cocktail}'")

    async def run(self):
      while self.clients.commandes or len(self.pic) > 0 or len(self.bar) > 0:
         await asyncio.sleep(0)
         await self.servir()

In [34]:
class Bariste(Employe):

    async def preparer(self):
        commande = self.pic.liberer()

        if commande is not None:
            self.log(f"je commence la fabrication de '{commande}'")

            for cocktail in commande:
                self.log(f"je prépare '{cocktail}'")

            self.bar.recevoir(commande)

            self.log(f"la commande {commande} est prête")

            return commande

        return None

    async def run(self):
        while True:
            await asyncio.sleep(0)

            commande = await self.preparer()

            if commande is None:
                break

In [35]:
async def main():

    le_pic = Pic(name="le_pic", verbose=False)
    le_bar = Bar(name="le_bar", verbose=False)

    les_clients = Clients("lundi")

    bob = Bariste(
        le_pic, le_bar, les_clients,
        name="bob", verbose=True
    )

    alice = Serveur(
        le_pic, le_bar, les_clients,
        name="alice", verbose=True
    )

    await asyncio.gather(
        les_clients.run(alice),
        alice.run(),
        bob.run()
    )

In [ ]:
logfile = "borabora.log"

logf = open(logfile, "w", encoding="utf-8")

print("\n---", file=logf, flush=True)

await main()

logf.close()